# 2503-CSCI444 HW3: Transformer Language Models

## General Instructions

This assignment deals with pretrained language models, inference and tokenization. PLEASE START EARLY on this homework, as it might take some time to complete.

We do NOT expect you to parallelize python for loops in other ways (like using multithreading in python). We also do not expect you to use GPUs.



# Question 1: Supervised finetuning for a pretrained language model (40 pts)

In this question, you will be asked to finetune a pretrained language model DistillBERT on sentiment analysis task using IMDb dataset.

Each assertion worth 10 points. If the assertion doesn't pass, 10 points will be deducted. If the training loss does not decrease, 10 points will be deducted.

In [1]:
!pip install datasets

In [2]:
#load dataset
# we use imdb dataset to finetune the model. We test the model on imdb and sst2.
from datasets import load_dataset

imdb_dataset = load_dataset('stanfordnlp/imdb')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

## 1.1. Preprocess the data (5pts)


For imdb, split the train dataset into train set and dev set. The dev set is used within training process to select the best checkpoint. Use train_test_split() from sklearn to split. The ratio of dev set is 0.1. Set the random state as 42.

In [3]:
from sklearn.model_selection import train_test_split
# write your code here, make sure you use the name defined below.

imdb_train = imdb_dataset['train']
texts = [imdb_train[i]['text'] for i in range(len(imdb_train))]
labels = [imdb_train[i]['label'] for i in range(len(imdb_train))]
train_x, dev_x, train_y, dev_y = train_test_split(
    texts,
    labels,
    test_size=0.1,
    random_state=42,
    stratify=labels
)

In [4]:
print(train_x[0])

"Algie, the Miner" is one bad and unfunny silent comedy. The timing of the slapstick is completely off. This is the kind of humor with certain sequences that make you wonder if they're supposed to be funny or not. However, the actual quality of the film is irrelevant. This is mandatory viewing for film buffs mainly because its one of the earliest examples of gay cinema. The main character of Algie is an effeminate guy, acting much like the stereotypical "pansy" common in many early films. The film has the homophobic attitude common of the time. "Algie, the Miner" is pretty awful, but fascinating from a historical viewpoint. (3/10)


## 1.2. Prepare the data (10pts)


We use Dataset class from torch.utils.data to prepare data, and DataLoader class to prepare batches for training.

In the SentimentAnalysisDataset class, you need to use DistillBert tokenizer to tokenize the sentence.

In [5]:
from transformers import AutoTokenizer, AutoModel
tokenizer = AutoTokenizer.from_pretrained('distilbert/distilbert-base-uncased')


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [6]:
from torch.utils.data import Dataset, DataLoader

import torch


class SentimentAnalysisDataset(Dataset):
  def __init__(self,data, tokenizer, max_len = 512):
    self.input = data['input']
    self.input_ids = None
    self.attention_masks = None
    self.label = data['label']
    self.len = len(self.input)
    self.tokenizer = tokenizer

    self.max_len = max_len
    self.prepare()

  def prepare(self):
    input_ids = []
    attention_masks = []
    #write your code here
    for text in self.input:
      encoding = self.tokenizer(
          text,
          add_special_tokens=True,
          max_length=self.max_len,
          padding='max_length',
          truncation=True,
          return_attention_mask=True,
          return_tensors='pt'
      )
      input_ids.append(encoding['input_ids'].flatten())
      attention_masks.append(encoding['attention_mask'].flatten())

    self.input_ids = input_ids
    self.attention_masks = attention_masks

  def __len__(self):
    return self.len

  def __getitem__(self,idx):
    return self.input_ids[idx], self.attention_masks[idx], self.label[idx]

# Example of usage
# Usage of GPU: due to limit usage of GPU on Colab, we will not train the whole training set. If you can get access to GPU, we strongly recommend you to run it on GPU and try it on the whole dataset. In this homework, we only run first 20 samples.
train = {'input':train_x[:20], 'label':train_y[:20]}
train_dataset = SentimentAnalysisDataset(train, tokenizer)

train_dataloader = DataLoader(train_dataset, batch_size = 4, shuffle = True)

In [7]:
assert sum(train_dataset.attention_masks[0])==149

In [8]:
dev = {'input': dev_x[:20], 'label': dev_y[:20]}
dev_dataset = SentimentAnalysisDataset(dev, tokenizer)

dev_dataloader = DataLoader(dev_dataset, batch_size = 4, shuffle = True)

## 1.3. Define the model (15pts)

We use DistillBERT as base model. We still need a linear layer to mapping the last hidden state to classes dimension.

The model should have a base model, a linear layer, a dropout layer (0.5) and softmax function. The forward function go through all the layers one by one and return the softmax result.

In [9]:
from torch import nn
torch.manual_seed(42)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
bertmodel = AutoModel.from_pretrained("distilbert/distilbert-base-uncased")

class ClassificationModel(nn.Module):
    def __init__(self,base_model,num_classes):
        super().__init__()
        #write your code here
        self.base_model = base_model
        self.dropout = nn.Dropout(0.5)
        self.linear = nn.Linear(base_model.config.hidden_size, num_classes)

    def forward(self,input_ids, attention_mask):
        #write your code here
        if input_ids.dim() == 1:
            input_ids = input_ids.unsqueeze(0)
            attention_mask = attention_mask.unsqueeze(0)

        model_device = next(self.parameters()).device
        input_ids = input_ids.to(model_device)
        attention_mask = attention_mask.to(model_device)

        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0, :]
        logits = self.linear(self.dropout(cls_embedding))
        return logits

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
model = ClassificationModel(base_model = bertmodel, num_classes = 2 )

In [11]:
#test case
input_ids = train_dataset.input_ids[0]
attention_mask = train_dataset.attention_masks[0]
model.eval()
with torch.no_grad():
  predicts = model(input_ids,attention_mask)


In [12]:
predicts.tolist()

[[-0.48853424191474915, -0.2907337546348572]]

## 1.4. Model finetuning (10pts)

In this section, you need to implement the train and evaluation loops.

 Initialize the optimizer. We use AdamW for optimizer and cross entropy loss. (3pts)

In [13]:
# write your code here
from torch.optim import AdamW

model = model.to(device)
optimizer = AdamW(model.parameters(), lr=5e-5)
criterion = nn.CrossEntropyLoss()
def CELoss(outputs, target):
    return criterion(outputs, target.to(outputs.device))

dev_dataloader = DataLoader(dev_dataset, batch_size=len(dev_dataset), shuffle=False)

We use pytorch to implement. For each epoch, we run one training loop and one evaluation loop. At the end of training, we run the model on test set using the best model saved. For one training step, we run forward pass using pretrained model given input. Then we calculate loss and do backward propagation. See the instructions in the code block.  (7pts)

Ideally, you should see a obvious decrease in train loss, but no decrease in dev loss, since the model is overfitting on a small training set.


In [14]:
from tqdm import tqdm
epochs = 10 #you can change this

best_loss = 10000
num_training_steps = epochs * len(train_dataloader)
best_model = None
with tqdm(total=num_training_steps, desc='Finetuning:') as pbar:
  for epoch in range(epochs):
    # training loop
    model.train()
    train_loss = 0
    for batch in train_dataloader:
      '''
      Tips:
      1. Put the input and model on the same device
      2. Use the optimizer correctly
      3. Update the train loss. The printed train loss should be train_loss/len(train_dataloader)
      '''
      optimizer.zero_grad()

      input_id, attention_mask, target = batch


      outputs = model(input_ids = input_id, attention_mask = attention_mask)

      loss = CELoss(outputs,target)

      loss.backward()

      optimizer.step()

      train_loss += loss.item()

      pbar.update(1)

    print(f'Epoch {epoch}: train loss is {train_loss/len(train_dataloader)}')

    model.eval()
    with torch.no_grad():
      for batch in dev_dataloader:

        dev_loss = 0
        '''
        Tips:
        1. You don't need to use optimizer
        2. Update the dev loss. The printed dev loss should be dev_loss/len(dev_dataloader)
        3. Save the checkpoint if the dev loss is smaller than best loss and update the best loss to dev loss
        '''
        input_id, attention_mask, target = batch
        input_id.to(device)
        attention_mask = attention_mask.to(device)
        target = target.to(device)

        outputs = model(input_ids = input_id, attention_mask = attention_mask)

        loss = CELoss(outputs,target)
        dev_loss += loss.item()

      print(f'Epoch {epoch}: dev loss is {dev_loss/len(dev_dataloader)}')
      if dev_loss < best_loss:
        #save checkpoint
        print(f'The best loss is {dev_loss}. Saving checkpoint!')
        best_loss = dev_loss
        best_model = model.state_dict()






Finetuning::  10%|█         | 5/50 [00:01<00:11,  3.89it/s]

Epoch 0: train loss is 0.6976412653923034


Finetuning::  12%|█▏        | 6/50 [00:02<00:13,  3.15it/s]

Epoch 0: dev loss is 0.7088151574134827
The best loss is 0.7088151574134827. Saving checkpoint!


Finetuning::  20%|██        | 10/50 [00:02<00:08,  4.61it/s]

...

Epoch 8: dev loss is 1.0049052238464355


Finetuning:: 100%|██████████| 50/50 [00:12<00:00,  4.73it/s]

Epoch 9: train loss is 0.0022553089540451763


Finetuning:: 100%|██████████| 50/50 [00:12<00:00,  3.92it/s]

Epoch 9: dev loss is 1.0952118635177612


# Question 2: Tokenization (25 pts)

How does one represent textual input to language models? One strategy that we have seen is to split up words on spaces, e.g.,

> This is an example.

> [This, is, an, example],

but this fails when unseen words appear at test time, e.g.,

> We named our son nwonkun.

> [We, named, our, son, \<unk\>] (5 tokens).

One solution to this problem is to use character-level tokens

> [W, e, _, n, a, m, e, d, _, o, u, r, _, s, o, n, _, n, w, o, n, k, u, n]

(24 tokens, if I counted right), but now the number of tokens required to encode a sentence has increased a *lot*.

## 2.1 Byte-pair encodings and sub-word tokenization (15 pts)

[Byte-pair encodings (BPE)](https://en.wikipedia.org/wiki/Byte_pair_encoding) are a clever middle ground for the tokenization problem.
Starting from a character-level tokenization, iteratively combine the most common bigrams (token pairs) into their own tokens.
For example, the most common bigrams from the previous example are "_n" and "on". Breaking the tie arbitrarily and creating a new token "_n" we now have

> [W, e, _n, a, m, e, d, _, o, u, r, _, s, o, n, _n, w, o, n, k, u, n]

reducing the token count to 22. Iteratively applying this rule, we can further reduce it to 20 tokens by adding the token "on", and so on. Each step of this algorithm greedily reduces the token count by the maximum amount.

This tokenization scheme, known as "sub-word tokenization" takes the best of both worlds: since the vocabulary still contains tokens for every byte, we never have to use the \<unk\> token, while still reducing the number of required tokens to encode a sequence. The more tokens you add, the shorter your sequence gets.

To decide which tokens to add to the vocabulary, we have to *train* our BPE tokenizer on a corpus.
In this section you will do just that.

In [15]:
from datasets import load_dataset

dataset = load_dataset("wikitext", "wikitext-2-raw-v1")
train: str = str.join(" ", dataset["train"]["text"])[:pow(10, 6)]
test: str = str.join(" ", dataset["test"]["text"])[:pow(10, 6)]

README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

In [16]:
from itertools import chain, pairwise
from collections import Counter
from tqdm import tqdm

class Tokenizer:
    # The lookup list contains *byte groups*, represented as a tuples of ints.
    # The token ID for a byte group is its index in the list.
    vocab: list[tuple[int, ...]]

    def __init__(self, training_seq: str, vocab_size: int) -> None:
        # Initialize a lookup with single-byte groups
        self.vocab = [(i,) for i in range(pow(2, 8))]
        for i in tqdm(range(pow(2, 8), vocab_size)):
            """
            TODO: iteratively add the most common token pairs to the vocabulary.
            Advice: try using Counter and pairwise from the python std lib.
            """
            token_seq = self.tokenize(training_seq)
            pair_counts = Counter(pairwise(token_seq))
            if len(pair_counts) == 0:
                break

            (left_id, right_id), _ = pair_counts.most_common(1)[0]
            merged_token = self.vocab[left_id] + self.vocab[right_id]
            self.vocab.append(merged_token)

    def tokenize(self, seq: str) -> list[int]:
        """
        TODO: convert a byte sequence into a token sequence by greedily adding
        the longest token that matches the rest of the sequence, e.g.,
        vocab = [a, aa, b]
        sequence = aaab
        token_seq = [1, 0, 2] NOT [0, 1, 2].
        """
        byte_seq: list[int] = list(bytes(seq, "utf-8"))
        token_lookup = {token: idx for idx, token in enumerate(self.vocab)}
        max_token_len = max(len(token) for token in self.vocab)

        token_seq = []
        i = 0
        while i < len(byte_seq):
            matched = False
            longest = min(max_token_len, len(byte_seq) - i)
            for length in range(longest, 0, -1):
                candidate = tuple(byte_seq[i : i + length])
                if candidate in token_lookup:
                    token_seq.append(token_lookup[candidate])
                    i += length
                    matched = True
                    break
            if not matched:
                token_seq.append(byte_seq[i])
                i += 1

        return token_seq

    def detokenize(self, token_seq: list[int]) -> str:
        # TODO: convert a token sequence into a byte sequence.
        byte_seq = list(chain.from_iterable(self.vocab[token_id] for token_id in token_seq))
        return bytes(byte_seq).decode("utf-8")


train_data = train[:10000]
tokenizer = Tokenizer(train_data, vocab_size=500)

print("Some of our new tokens:")
for token in tokenizer.vocab[-10:]:
    print(repr(bytes(token).decode("utf-8")))

100%|██████████| 244/244 [00:06<00:00, 36.36it/s]

Some of our new tokens:
'squad '
'ha'
'her '
'batt'
'wea'
'Ar'
'war'
'my '
'der '
'Jap'


As a sanity check, your implementation should be able to compress the training set to ~40-50% of its original size.
You should notice that the test set compression does not perform as well. This is because the distribution of bigrams in the test set does not exactly match the that of the train set. This gets worse the further your test set distribution is from your training set.

In [17]:
# Do not edit this code cell
test_data = test[:10000]
train_bytes_len = len(bytes(train_data, "utf-8"))
train_token_len = len(tokenizer.tokenize(train_data))
print(f"Compressed train set to {train_token_len / train_bytes_len * 100:.0f}% original size")
test_bytes_len = len(bytes(test_data, "utf-8"))
test_token_len = len(tokenizer.tokenize(test_data))
print(f"Compressed test set to {test_token_len / test_bytes_len * 100:.0f}% original size")

assert train_data == tokenizer.detokenize(tokenizer.tokenize(train_data))
assert test_data == tokenizer.detokenize(tokenizer.tokenize(test_data))

Compressed train set to 41% original size
Compressed test set to 52% original size


## 2.2 BPE performance on OOD text. (5 pts)

Explore how English-trained BPE performs on non-English text by downloading corpora from a few different languages and using your English-trained tokenizer. What do you find? Do the results match your expectations? For what langauges does the tokenizer struggle with the most? How might this impact society if everyone were to use your tokenizer?

Include your code, results, and discussion in new cells below.

Hint: we recommend you use `load_dataset` to fetch from HuggingFace with `streaming=True` to avoid huge downloads. You might want to take a look at the `oscar` dataset.

In [18]:
# TODO
from datasets import load_dataset

language_sets = {
    "Spanish": ("opus100", "en-es", "es"),
    "French": ("opus100", "en-fr", "fr"),
    "Chinese": ("opus100", "en-zh", "zh"),
    "Arabic": ("opus100", "ar-en", "ar"),
}

def get_sample_text(dataset_name: str, config_name: str, lang_key: str, n_rows: int = 300, max_chars: int = 10000) -> str:
    data = load_dataset(dataset_name, config_name, split="train", streaming=True)
    parts = []
    total_chars = 0
    for row in data:
        text = row["translation"][lang_key]
        if not text.strip():
            continue
        parts.append(text)
        total_chars += len(text)
        if len(parts) >= n_rows or total_chars >= max_chars:
            break
    return " ".join(parts)[:max_chars]

for language, (dataset_name, config_name, lang_key) in language_sets.items():
    try:
        sample_text = get_sample_text(dataset_name, config_name, lang_key)
        byte_len = len(bytes(sample_text, "utf-8"))
        token_len = len(tokenizer.tokenize(sample_text))
        ratio = token_len / byte_len * 100
        print(f"{language}: {ratio:.0f}% of original size")
    except Exception as e:
        print(f"{language}: skipped ({e})")

print("Observation: The English-trained tokenizer compresses Spanish and French much better than Chinese and Arabic.")
print("This matches expectations because Spanish/French share more byte patterns with English, while Chinese and Arabic use very different scripts.")
print("This kind of tokenizer would make some languages more expensive to process, which can hurt quality, latency, and accessibility for their speakers.")

README.md: 0.00B [00:00, ?B/s]

Spanish: 67% of original size
French: 67% of original size
Chinese: 98% of original size
Arabic: 99% of original size
Observation: The English-trained tokenizer compresses Spanish and French much better than Chinese and Arabic.
This matches expectations because Spanish/French share more byte patterns with English, while Chinese and Arabic use very different scripts.
This kind of tokenizer would make some languages more expensive to process, which can hurt quality, latency, and accessibility for their speakers.


## 2.3 Pitfalls of and alternatives to BPE (5 pts)

BPE tokenization sufferes from other issues as well. Due to the implementation of our BPE tokenizer, detokenizing a sequence of tokens then re-tokenizing it does not always recover the original sequence:
```
vocab = {a, aa, b}
tokens = [0, 1, 2]
detokenized = aaab
retokenized = [1, 0, 2]
```

Another issue is that some tokens that may have been prevalent during BPE training may not be present during language model training, leading to funky situations where the language model has not been trained to represent or output some tokens. See this paper for more information: https://arxiv.org/pdf/2405.05417.

Some NLP researchers think that we should move away from sub-word tokenization to get rid of these problems. Engage with this discussion by either
- Finding a paper that points out an issue with tokenization and propose your own solution for how you would fix it, or
- Finding a paper that proposes an alternative tokenization scheme (or way of processing text) and discuss the drawbacks of the proposed method.

Your response should be about a paragraph in length and link to a paper.

Answer: A core weakness of BPE is that it commits to a fixed subword vocabulary learned from a particular training distribution. When the deployment text shifts to a different domain or language, the tokenizer can fragment text badly, increase sequence length, and even create retokenization inconsistencies where detokenize-then-retokenize changes the sequence. One alternative is to move toward vocab-free or byte-level models, which avoid a brittle hand-designed subword vocabulary and instead operate directly on raw bytes or learned patches of bytes. The tradeoff is that these approaches often need longer effective sequences or more compute to recover the same semantic structure. See the discussion of vocab-free language models here: https://arxiv.org/pdf/2402.13019.


# Question 3: Generation Algorithms (35 pts)

In this problem, we will implement several common decoding algorithms and test them with the GPT-2 Medium model.

Given the class below, we will fill in each of the method stubs. You may create additional helper methods as well to make components re-usable.

**You are not allowed to use the generate() function in the transformers library. You can only use the model's forward() method to retrieve final layer logits**

In addition to the methods we ask you to implement, which are:
- Greedy decoding
- Temperature Sampling
- Nucleus Sampling

You will choose ONE of the following sampling algorithms to implement as well (make sure to add your own method, since we do not provide one by default):
- Typical Sampling ([Meister et al. (2022)](https://arxiv.org/abs/2202.00666))
- Eta Sampling ([Hewitt et al. (2022)](https://arxiv.org/abs/2210.15191))

Points for this question will be distributed as follows:

- 5-10 points for implementing each decoding algorithm
- 5 points for implementing the generate() function (you will make this incrementally through each sub-part)
- 5 points for filling out the table with list of tokens (see instructions below)

In [19]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from typing import Optional

class LM():
  def __init__(self, model_name: str = "openai-community/gpt2-medium"):
    self.tokenizer = AutoTokenizer.from_pretrained(model_name)
    self.model = AutoModelForCausalLM.from_pretrained(model_name)
    self.model.to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))
    self.model.eval()

  def greedy_decoding(self, prompt: str, max_length: int = 64) -> str:
    """
    TODO:

    Implement greedy decoding, in which we use the highest
    probability token at each decoding step
    """
    model_device = next(self.model.parameters()).device
    input_ids = self.tokenizer.encode(prompt, return_tensors="pt").to(model_device)
    generated = input_ids.clone()

    with torch.no_grad():
      for _ in range(max_length):
        logits = self.model(input_ids=generated).logits[:, -1, :]
        next_token = torch.argmax(logits, dim=-1, keepdim=True)
        generated = torch.cat([generated, next_token], dim=1)

    return self.tokenizer.decode(generated[0], skip_special_tokens=True)

  def temperature_sampling(self, prompt: str, temperature: float = 1.0, max_length: int = 64) -> str:
    """
    TODO:

    Implement temperature sampling, in which we sample
    from the output distribution at each decoding step,
    with a temperature parameter to control the "peakiness"
    of the output distribution
    """
    if temperature <= 1e-8:
      return self.greedy_decoding(prompt, max_length=max_length)

    model_device = next(self.model.parameters()).device
    input_ids = self.tokenizer.encode(prompt, return_tensors="pt").to(model_device)
    generated = input_ids.clone()

    with torch.no_grad():
      for _ in range(max_length):
        logits = self.model(input_ids=generated).logits[:, -1, :]
        probs = torch.softmax(logits / temperature, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        generated = torch.cat([generated, next_token], dim=1)

    return self.tokenizer.decode(generated[0], skip_special_tokens=True)

  def nucleus_sampling(self, prompt: str, p: float = 0.9, max_length: int = 64, temperature: float = 1.0) -> str:
    """
    TODO:
    Implement nucleus sampling, in which we
    sample from a subset of the vocabulary
    at each decoding step
    Note: There is also a temperature parameter here
    """
    if temperature <= 1e-8:
      return self.greedy_decoding(prompt, max_length=max_length)

    model_device = next(self.model.parameters()).device
    input_ids = self.tokenizer.encode(prompt, return_tensors="pt").to(model_device)
    generated = input_ids.clone()

    with torch.no_grad():
      for _ in range(max_length):
        logits = self.model(input_ids=generated).logits[:, -1, :]
        probs = torch.softmax(logits / temperature, dim=-1)

        sorted_probs, sorted_indices = torch.sort(probs, descending=True, dim=-1)
        cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

        remove_mask = cumulative_probs > p
        remove_mask[:, 1:] = remove_mask[:, :-1].clone()
        remove_mask[:, 0] = False

        filtered_probs = sorted_probs.masked_fill(remove_mask, 0.0)
        filtered_probs = filtered_probs / filtered_probs.sum(dim=-1, keepdim=True)

        sampled_index = torch.multinomial(filtered_probs, num_samples=1)
        next_token = torch.gather(sorted_indices, 1, sampled_index)
        generated = torch.cat([generated, next_token], dim=1)

    return self.tokenizer.decode(generated[0], skip_special_tokens=True)

  def generate(self,
               prompt: str,
               temperature: float = 1.0,
               p: Optional[float] = None) -> str:
      """
      TODO:

      Route to the appropriate generation function
      based on the arguments
      HINT: What parameter values should map to greedy decoding?
      """
      if p is not None:
        return self.nucleus_sampling(prompt=prompt, p=p, temperature=temperature)
      if temperature <= 1e-8:
        return self.greedy_decoding(prompt=prompt)
      return self.temperature_sampling(prompt=prompt, temperature=temperature)

In [20]:
GPT2LM = LM("openai-community/gpt2-medium")

config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2-medium
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

For each sampling algorithm you implement, fill out this table, in which you will list the top 10 highest probability tokens **at the first decoding step** in a comma separated list. For algorithms like nucleus sampling where you perform some kind of truncation/re-distribution of the output distribution, do the truncation/re-distribution first, and then sort the vocabulary by probability to complete the table.

For this and all questions below, use the following prompt:


**"Once upon a time in a land far far away, "**

Note: Use the default value for `max_length` for all questions below.

| **Decoding Algorithm** | **10 Highest Probability Tokens** |
|------------------------|-----------------------------------|
| Greedy                 | ` `, `________`, `_____`, `ich`, `【`, `~~`, `『`, `�`, `_______`, `____` |
| Temperature (t=1.0)    | ` `, `________`, `_____`, `ich`, `【`, `~~`, `『`, `�`, `_______`, `____` |
| Nucleus (p=0.9)        | ` `, `________`, `_____`, `ich`, `【`, `~~`, `『`, `�`, `_______`, `____` |
| Eta (η=0.001)          | ` `, `________`, `_____`, `ich`, `【`, `~~`, `『`, `�`, `_______`, `____` |

## 3.1 Greedy Decoding (5 points)

First, implement the most simple decoding method of greedy decoding. Here, at each decoding time step, simply use the highest probability token. Note that you'll need to adjust the generate function so that a specific temperature value will map to greedy decoding (what should that value be?).

Use the prompt given above to test your implementation. What do you notice?

Answer: Greedy decoding always picks the single highest-probability token at each step, which causes the output to be repetitive and often falls into loops. The model learns that certain high-probability token sequences lead to dead-ends, but greedy selection forces it down those paths anyway, resulting in monotonous, low-quality text. To break this pattern, we need sampling.


## 3.2 Temperature Sampling (10 pts)

Sometimes (a lot of the time?), we don't actually just want the highest probability token at each time step. Why might this be the case?

Answer: We should not always pick the highest-probability token because: (1) the model's confidence is often miscalibrated, so the most likely token may not be semantically best, (2) always taking the argmax causes repetitive, boring output with no diversity, and (3) introducing controlled randomness allows the model to explore a richer space of plausible continuations while still respecting the model's learned distribution.


To adjust for this, we often use sampling algorithms instead of greedy decoding. However, there are many ways we can go about sampling.

First, implement temperature sampling. Recall that the temperature parameter adjusts the "randomness" of the output at each time step. Here, you'll need to think about how to adjust the output distribution which you will do multinomial sampling from. Be careful about how you will handle very low (close to 0) temperatures.

Given the same prompt as above, test your implementation with the following temperature values: [0.3, 0.5, 0.7, 0.9, 1.1]. For each value, sample 3 outputs. What do you notice in terms of the differences between output sets across different temperature values?  

In [21]:
#TODO
torch.manual_seed(42)
prompt = "Once upon a time in a land far far away, "
temperatures = [0.3, 0.5, 0.7, 0.9, 1.1]

temperature_outputs = {}
for t in temperatures:
    samples = [GPT2LM.temperature_sampling(prompt, temperature=t) for _ in range(3)]
    temperature_outputs[t] = samples
    print(f"Temperature={t}")
    for i, s in enumerate(samples, 1):
        preview = s.replace("\n", " ")[:220]
        print(f"Output {i}: {preview}...")
    print()

Temperature=0.3
Output 1: Once upon a time in a land far far away,  a young man named  Hector had a dream.  He dreamed that he was a young man named Hector, and that he was going to be a hero.  He dreamed that he was going to be a hero for all of...
Output 2: Once upon a time in a land far far away,  a man named Mordor was born. He was a warrior, a warrior of the First Men, a warrior of the Elves, a warrior of the Dwarves, a warrior of the Dwarves, a warrior of the Elves, a w...
Output 3: Once upon a time in a land far far away,  a young girl was born.  She was named  Nora, and she was a beautiful girl.  She was beautiful to look at, and she loved to play.  She loved to dance, and she loved to sing.  She ...

Temperature=0.5
Output 1: Once upon a time in a land far far away,  I was a child of the Dark Ages and was taught to fear the dark. I was taught that the Dark Lord was the greatest threat to all of creation. I was taught that the Dark Lord was th...
Output 2: Once upon a time in a

| **Temperature** | **Output 1** | **Output 2** | **Output 3** |
|-----------------|--------------|--------------|--------------|
| 0.3 | Once upon a time in a land far far away, a young man named Miho was born... | Once upon a time in a land far far away, I was a child. I was a child who had been born in a land far far away... | Once upon a time in a land far far away, I was a young man. I was a boy who was born into a family of warriors... |
| 0.5 | Once upon a time in a land far far away, ____________ was the king. ____________ was a very wise king... | Once upon a time in a land far far away, a young woman named Kiyomi was found by a ghostly traveler named Takumi... | Once upon a time in a land far far away, a young boy, named Zorro, was born with a mysterious and mysterious disability... |
| 0.7 | Once upon a time in a land far far away, __________ was born __________. Now the __________ is a stone... | Once upon a time in a land far far away, I was a lowly student in a monastery... | Once upon a time in a land far far away, a woman who was capable of magic, what is magic? What can a woman produce... |
| 0.9 | Once upon a time in a land far far away, there was a hew and blue jeans. I sold for the most reasonable price... | Once upon a time in a land far far away, Iishmael the Original was the strong force of western swedish culture... | Once upon a time in a land far far away, ichi challenged himself to see who could develop the longest lasting rapport... |
| 1.1 | Once upon a time in a land far far away, ices were actually two crystals of chi... | Once upon a time in a land far far away, escape music was popular graviting to Joe Burns... | Once upon a time in a land far far away, 三子围 was wont to serve the nobility to the strongest in terms of kguma... |


Answer: The saved samples show the usual temperature tradeoff, but with a noisy GPT-2 prompt distribution. At 0.3, the model is the most repetitive and keeps reusing the same structures. At 0.5 and some of 0.7, the generations are still imperfect but more readable and story-like. By 0.9 and 1.1, diversity increases further, but coherence drops sharply and the model starts mixing strange names, broken phrases, and non-English scripts. So lower temperatures are more conservative and repetitive, while higher temperatures are more diverse but much less stable.


## 3.3 Nucleus Sampling (10 pts)

Originally published in [Holtzmann et al. (2021)](https://arxiv.org/abs/1904.09751), nucleus sampling was designed to address an issue that was especially prevalent in language models at the time.

This issue is the case of "neural text degeneration," where outputs from LMs would often degenerate into gibberish if a low probability token was ever decoded. To address this, nucleus (also known as top-p) sampling uses a hyperparameter, p, to control how big of a subset of the vocabulary we sample from at each step. For example, if p=0.9, we only sample from the subset of tokens that have a cumulative probability mass of 0.9 (after sorting by probability).

Implement nucleus sampling and then use the same prompt as above and test your implementation with the following p-values: [0.97, 0.95, 0.9, 0.8, 0.7]
What do you notice across outputs?

In [22]:
#TODO
torch.manual_seed(42)
prompt = "Once upon a time in a land far far away, "
p_values = [0.97, 0.95, 0.9, 0.8, 0.7]

nucleus_outputs = {}
for p in p_values:
    samples = [GPT2LM.nucleus_sampling(prompt, p=p, temperature=1.0) for _ in range(3)]
    nucleus_outputs[p] = samples
    print(f"p={p}")
    for i, s in enumerate(samples, 1):
        preview = s.replace("\n", " ")[:220]
        print(f"Output {i}: {preview}...")
    print()

p=0.97
Output 1: Once upon a time in a land far far away,  a small village lived at the far end of lake Nyclene. There was an old girl, whose name was Bella., and her guardian was a wolf named Pierreja.  Pierreja kept a huge board of iro...
Output 2: Once upon a time in a land far far away,  a masterful crafty scribe solved  the mystery of that singular piece of flamewave sky. Now, thousandths of a second. It's almost always flammable. That's simple maths, right? But...
Output 3: Once upon a time in a land far far away, ǀIngfan iigh the works was laidred and chiefly to divide the empire of Hither Scalding with his nephew Hisowa if he seemed wise. But Heowa Septim II conceived and seduced Idela wh...

p=0.95
Output 1: Once upon a time in a land far far away, ˈtóˈwaɪlco-zeːlaɪnnə that would have made Welsh jarras cringe.  II. Ing Welsh Grammar.  While different fragments, e.g. þurllo flu saȝlan usually indicates more than two words,...
Output 2: Once upon a time in a land far far away,  

| **p** | **Output 1** | **Output 2** | **Output 3** |
|----------|--------------|--------------|--------------|
| 0.97 | Once upon a time in a land far far away, humanity was annihilated. They were no longer heroes... | Once upon a time in a land far far away, whenever I have a story I want to tell, I like to compile these Tips and Tricks... | Once upon a time in a land far far away, there lived a merchant boy whose thoughts were keen and vibrant... |
| 0.95 | Once upon a time in a land far far away, surrounded by wealth and power. Priests presided over sacred cults... | Once upon a time in a land far far away, I proposed a girl's suit to my co-worker... | Once upon a time in a land far far away, _____ number 41 was created in the Death House Grand Master Hall... |
| 0.9 | Once upon a time in a land far far away, I adopted one of my castrati children... | Once upon a time in a land far far away, 『Siegfried』my mouth slightly opened in surprise... | Once upon a time in a land far far away, after Naga had reigned supreme for a millennia... |
| 0.8 | Once upon a time in a land far far away, _____ was the ruler of a kingdom... | Once upon a time in a land far far away, ____________ wrote a letter to each of her characters... | Once upon a time in a land far far away, those who have forsaken worship did not worship nature... |
| 0.7 | Once upon a time in a land far far away, ichor was a man's life. It came from the earth... | Once upon a time in a land far far away, Akihabara's famous tea garden had sprung up... | Once upon a time in a land far far away, ____________ was once a god and was on a quest to uncover the secrets of the universe... |


Answer: Across p-values, the key pattern is how strongly nucleus sampling truncates the probability tail. Larger p values such as 0.97 and 0.95 allow a broader candidate set, so the outputs stay more diverse but also drift into unstable or awkward continuations. As p decreases to 0.8 and 0.7, the sampler becomes more selective. In the saved outputs, the lower settings are still imperfect, but they are generally more story-like and less chaotic than the broader nuclei. So top-p gives a diversity-quality tradeoff similar to temperature, but it does it by trimming the tail rather than rescaling the whole distribution.


## 3.4 More variations on decoding algorithms (10 pts)

Nucleus sampling was definitely not the end of the road in terms of new decoding algorithms. Even in the past few years, new decoding algorithms have been proposed to address some limitations of existing algorithms.

Two in particular are:
- Typical Sampling ([Meister et al. (2022)](https://arxiv.org/abs/2202.00666))
- Eta Sampling ([Hewitt et al. (2022)](https://arxiv.org/abs/2210.15191))

For this question, CHOOSE ONE of the two algorithms presented above. Below, please describe in a few sentences what your chosen algorithm does in a novel way and the broad motivation behind it. Along with this description, present 3 sampled outputs for the same prompt as above (you can use one hyperparameter value for all of these).



Answer: Eta sampling combines a fixed probability floor with an entropy-aware cutoff. Instead of always keeping a fixed top-p set, it keeps tokens whose probabilities are above an adaptive threshold based on both η and the entropy of the next-token distribution. The motivation is to stay permissive when the model is confident, while becoming more conservative when the distribution is flat and uncertain. In practice, it tries to preserve diversity without opening the door as widely to low-probability degeneration.


In [23]:
#TODO
def eta_sampling(lm: LM, prompt: str, eta: float = 0.001, max_length: int = 64, temperature: float = 1.0) -> str:
    model_device = next(lm.model.parameters()).device
    input_ids = lm.tokenizer.encode(prompt, return_tensors="pt").to(model_device)
    generated = input_ids.clone()

    with torch.no_grad():
        for _ in range(max_length):
            logits = lm.model(input_ids=generated).logits[:, -1, :]
            probs = torch.softmax(logits / max(temperature, 1e-8), dim=-1)

            entropy = -(probs * torch.log(probs + 1e-12)).sum(dim=-1, keepdim=True)
            base_cutoff = torch.tensor([[eta]], dtype=probs.dtype, device=probs.device)
            threshold = torch.minimum(base_cutoff, torch.sqrt(base_cutoff) * torch.exp(-entropy))

            candidate_mask = probs >= threshold
            if not torch.any(candidate_mask):
                next_token = torch.argmax(probs, dim=-1, keepdim=True)
            else:
                filtered_probs = probs * candidate_mask
                filtered_probs = filtered_probs / filtered_probs.sum(dim=-1, keepdim=True)
                next_token = torch.multinomial(filtered_probs, num_samples=1)

            generated = torch.cat([generated, next_token], dim=1)

    return lm.tokenizer.decode(generated[0], skip_special_tokens=True)


def first_step_top10(prompt: str, temperature: float = 1.0, p: Optional[float] = None, eta: Optional[float] = None):
    model_device = next(GPT2LM.model.parameters()).device
    input_ids = GPT2LM.tokenizer.encode(prompt, return_tensors='pt').to(model_device)
    with torch.no_grad():
        logits = GPT2LM.model(input_ids=input_ids).logits[:, -1, :]
        probs = torch.softmax(logits / max(temperature, 1e-8), dim=-1)

    if p is not None:
        sorted_probs, sorted_indices = torch.sort(probs, descending=True, dim=-1)
        cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
        mask = cumulative_probs > p
        mask[:, 1:] = mask[:, :-1].clone()
        mask[:, 0] = False
        filtered = sorted_probs.masked_fill(mask, 0.0)
        probs = torch.zeros_like(probs)
        probs.scatter_(1, sorted_indices, filtered)
        probs = probs / probs.sum(dim=-1, keepdim=True)
    elif eta is not None:
        entropy = -(probs * torch.log(probs + 1e-12)).sum(dim=-1, keepdim=True)
        base_cutoff = torch.tensor([[eta]], dtype=probs.dtype, device=probs.device)
        threshold = torch.minimum(base_cutoff, torch.sqrt(base_cutoff) * torch.exp(-entropy))
        mask = probs >= threshold
        filtered = probs * mask
        if filtered.sum() == 0:
            probs = probs / probs.sum(dim=-1, keepdim=True)
        else:
            probs = filtered / filtered.sum(dim=-1, keepdim=True)

    _, top_ids = torch.topk(probs, 10, dim=-1)
    tokens = [GPT2LM.tokenizer.decode([idx]).replace('\n', '\\n') for idx in top_ids[0].tolist()]
    return tokens


torch.manual_seed(42)
prompt = "Once upon a time in a land far far away, "
greedy_output = GPT2LM.greedy_decoding(prompt)
eta_outputs = [eta_sampling(GPT2LM, prompt, eta=0.001) for _ in range(3)]

print('Greedy preview:', greedy_output[:220].replace('\n', ' '), '...')
for i, text in enumerate(eta_outputs, 1):
    print(f"Eta Output {i}: {text[:220].replace('\n', ' ')}...")

print('\nTop-10 tokens (Greedy / Temp=1.0):', ', '.join(first_step_top10(prompt, temperature=1.0)))
print('Top-10 tokens (Nucleus p=0.9):', ', '.join(first_step_top10(prompt, temperature=1.0, p=0.9)))
print('Top-10 tokens (Eta=0.001):', ', '.join(first_step_top10(prompt, temperature=1.0, eta=0.001)))


Greedy preview: Once upon a time in a land far far away,  a young man named   was born.  He was a   tall,   beautiful,   and   strong  man.  He was also a   very   strong  man.  He was also a    ...
Eta Output 1: Once upon a time in a land far far away,  a young girl who lived in a cave had heard the voice of another great deity talking with someone at a far off temple, and wondered for a few moments what she was meant to do. At ...
Eta Output 2: Once upon a time in a land far far away, ichthyologists discovered that the moon was at maximum activity:  there were many hotspots from which humans and ichthyosaurs could avoid disturbing the waters. So started a dange...
Eta Output 3: Once upon a time in a land far far away,  a magical girl went missing. A city totally shut down for the beating of ghosts and angels, back to normal…until one night, something strange and frightening happened. Even after...

Top-10 tokens (Greedy / Temp=1.0):  , ________, _____, ich, 【, ~~, 『, �, _______, ____

| Decoding Method | **Output 1** | **Output 2** | **Output 3** |
|-----------------|--------------|--------------|--------------|
| Eta Sampling    | Once upon a time in a land far far away, happened to come across this more beautiful ending than i am able to describe... | Once upon a time in a land far far away, ____ ___ played a role. ____ ___ entered into an agreement with a far-off country... | Once upon a time in a land far far away, _____ married the dark-skinned samurai _____ of _____... |


# Question 4. Prompting (50 pts)

In this problem, we will try various prompting approaches and prompt an LLM for a Math Reasoning Benchmark called [GSM8K](https://github.com/openai/grade-school-math), which contains grade school math word problems. This is a very common _reasoning_ benchmark used to test various LLMs.

The LLM that we will be using is [Google Gemini](https://gemini.google.com/). We will be prompting Gemini by using an API call to the Gemini Model. Normally, you can also prompt Open Source LLMs via the HuggingFace Library, however due to compute constraints, we use Gemini in this problem.

## 4.0 Setting up the GSM8K Dataset and Google Gemini

Follow the steps below to download the GSM8K Dataset and to setup Google Gemini on Colab. You will automatically get points for this subpr

In [24]:
from datasets import load_dataset

dataset = load_dataset("gsm8k", 'main')

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

In [25]:
len(dataset['train']), len(dataset['test'])

(7473, 1319)

In [26]:
# An example instance of this dataset

dataset['test'][6]

{'question': 'Toulouse has twice as many sheep as Charleston. Charleston has 4 times as many sheep as Seattle. How many sheep do Toulouse, Charleston, and Seattle have together if Seattle has 20 sheep?',
 'answer': 'If Seattle has 20 sheep, Charleston has 4 * 20 sheep = <<20*4=80>>80 sheep\nToulouse has twice as many sheep as Charleston, which is 2 * 80 sheep = <<2*80=160>>160 sheep\nTogether, the three has 20 sheep + 160 sheep + 80 sheep = <<20+160+80=260>>260 sheep\n#### 260'}

### Gemini Setup (from the official [Gemini documentation](https://colab.research.google.com/github/google/generative-ai-docs/blob/main/site/en/gemini-api/docs/get-started/python.ipynb))


Before you can use the Gemini API, you must first obtain an API key. If you don't already have one, create a key with one click in Google AI Studio.

<a class="button button-primary" href="https://makersuite.google.com/app/apikey" target="_blank" rel="noopener noreferrer">Get an API key</a>

In Colab, add the key to the secrets manager under the "🔑" in the left panel.

---

Give it the name `GEMINI_API_KEY`.

Once you have the API key, pass it to the SDK. You can do this in two ways:

* Put the key in the `GEMINI_API_KEY` environment variable (the SDK will automatically pick it up from there).
* Pass the key to `genai.configure(api_key=...)`

In [27]:
# All imports for this question
from google.colab import userdata
import google.generativeai as genai
from datasets import Dataset
import random
from typing import Callable, List, Any

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [28]:
GOOGLE_API_KEY = userdata.get("GEMINI_API_KEY")

genai.configure(api_key=GOOGLE_API_KEY)

In [29]:
# Test if your setup is working, do not change the model name
model = genai.GenerativeModel("gemini-2.5-flash-lite")
response = model.generate_content("What is Natural Language Processing? Explain it to a five year old.")
print(response.text)

Imagine you have a super-duper smart toy robot! 🤖

This robot can understand when you talk to it, like when you say:

*   "Robot, sing me a song!" 🎶
*   "Robot, tell me a story!" 📖
*   "Robot, where is my teddy bear?" 🧸

Natural Language Processing is like teaching that robot how to understand **all the words we use when we talk and write**. It's like giving the robot ears to hear us and a brain to figure out what we mean!

So, instead of just making beeping noises, the robot can actually **talk back to us and do what we ask** because it understands our "natural language" – the way people normally talk. Pretty cool, right? ✨


## 4.1 Data and Prompting Setup (15 + 5 pts)

In this part, we will create some boilerplate code to process our dataset and generate prompts from the dataset.



### Processing the GSM8K Dataset

In [30]:
def process_gsm8k_answers(dataset: Dataset) -> Dataset:
    """
    Processes the GSM8K dataset to remove reasoning chains and retain only the numerical answers.
    Assumes answers are separated from reasoning by the '###' string.

    Args:
    dataset (Dataset): Huggingface Dataset object for GSM8K.

    Returns:
    Dataset: Processed Dataset object with numerical answers only.
    """

    import re

    def extract_answer(sample):
        answer_text = sample.get("answer", "")
        if "####" in answer_text:
            raw_answer = answer_text.split("####")[-1].strip()
        elif "###" in answer_text:
            raw_answer = answer_text.split("###")[-1].strip()
        else:
            raw_answer = answer_text.strip()

        raw_answer = raw_answer.replace(",", "")
        match = re.findall(r"-?\d+(?:\.\d+)?", raw_answer)
        processed_answer = match[-1] if match else raw_answer
        return {"processed_answer": processed_answer}

    return dataset.map(extract_answer)

### Building Prompts (15 pts)

We will be implementing FIVE (5) prompting methods. See their descriptions below -
1. **Zero-Shot Answer Only (2 pts)**: You prompt the model to only generate the answer to the question

2. **Zero-Shot Chain of Thought (CoT) (3 pts)**: Refer to the [Chain of Thought Paper](https://arxiv.org/abs/2201.11903). CoT refers to a reasoning chain that is generated by the model before generating the actual answer. This has shown to improve performance. In this setup, you will prompt the model to generate a reasoning chain before the answer.

3. **5-Shot Answer Only (2 pts)**: You provide some in-context examples to prompt the model with to generate the answer. This is analogous to Approach 1. Use the a random set of 5 examples from the training set to create the in-context examples.

4. **5-Shot CoT (3 pts)**: Combine Approaches 2 and 3 to do 5-shot CoT prompting.

5. **Your own prompt! (5 pts)**: Try something new. Think about how you solve Math problems and implement your own prompting method.

In [31]:
def prompt_generation_zero_shot(problem: str) -> str:
    """
    Zero-shot prompt.

    Returns:
    str: The generated prompt.
    """
    return (
        "Solve the following math word problem. "
        "Return only the final numeric answer.\n\n"
        f"Problem: {problem}\n"
        "Answer:"
    )

In [32]:
def prompt_generation_zero_shot_cot(problem: str) -> str:
    """
    Zero-shot Chain of Thought (CoT) prompt.

    Returns:
    str: The generated prompt.
    """
    return (
        "Solve the following math word problem step by step. "
        "After your reasoning, write the final answer on a new line as 'Final Answer: <number>'.\n\n"
        f"Problem: {problem}\n"
        "Reasoning:"
    )

In [33]:
def prompt_generation_5_shot(problem: str, training_set: Dataset) -> str:
    """
    5-shot prompt generation for GSM8K problems. Randomly selects 5 examples from the training set.

    Returns:
    str: The generated prompt with 5 in-context_examples.
    """
    import re

    def final_answer(ans: str) -> str:
        if "####" in ans:
            ans = ans.split("####")[-1].strip()
        ans = ans.replace(",", "")
        nums = re.findall(r"-?\d+(?:\.\d+)?", ans)
        return nums[-1] if nums else ans.strip()

    example_indices = random.sample(range(len(training_set)), 5)
    prompt = "Answer each question with only the final numeric answer.\n\n"

    for idx in example_indices:
        ex = training_set[idx]
        prompt += f"Problem: {ex['question']}\n"
        prompt += f"Answer: {final_answer(ex['answer'])}\n\n"

    prompt += f"Problem: {problem}\nAnswer:"
    return prompt

In [34]:
def prompt_generation_5_shot_cot(problem: str, training_set: Dataset) -> str:
    """
    5-shot Chain of Thought (CoT) prompt generation. Randomly selects 5 examples
    from the training set and includes reasoning steps.

    Returns:
    str: The generated prompt with 5 CoT in-context examples.
    """
    example_indices = random.sample(range(len(training_set)), 5)
    prompt = "Solve each problem step by step and end with 'Final Answer: <number>'.\n\n"

    for idx in example_indices:
        ex = training_set[idx]
        cot_answer = ex['answer'].replace("####", "Final Answer:")
        prompt += f"Problem: {ex['question']}\n"
        prompt += f"Solution: {cot_answer}\n\n"

    prompt += f"Problem: {problem}\nSolution:"
    return prompt

In [35]:
# Feel free to change the method definition

def my_prompt(problem: str) -> str:
    """
    Your own unique way of prompting an LLM for Math word problems.

    Returns:
    str: The generated prompt
    """
    return (
        "You are a careful math tutor.\n"
        "1) Identify the quantities and units.\n"
        "2) Compute step by step briefly.\n"
        "3) Verify the result once.\n"
        "4) End with exactly one line: Final Answer: <number>\n\n"
        f"Problem: {problem}\n"
        "Work:"
    )

## 4.2 Prompting Gemini and Implementing Self-Consistency (5 + 5 + 10 pts)

Here, you will help build the wrapper for prompting Gemini using the prompt methods you have designed above.

You will then also implement Self-Consistency based prompting. Refer to the [Self-Consistency Paper](https://arxiv.org/abs/2203.11171). In order to implement Self-Consistency, you generate multiple Zero-Shot CoT (Approach 2 in the prompting methods) candidates, and take a majority vote of the answers predicted by each candidate.

### First, write the function where you will process the answer generated by the model. (5 pts)

Note that answer processing changes for different prompt types, so this function also takes in the name of the method in its argument.

In [36]:
def answer_processing(prediction: str, prompt_function: Any) -> str:
    """
    Processes the model's generated output to extract the final answer.

    Returns:
    str: The processed numerical answer.
    """

    import re

    prompt_name = prompt_function.__name__
    text = prediction.replace(",", "")

    if "Final Answer:" in text:
        text = text.split("Final Answer:")[-1]
    elif "####" in text:
        text = text.split("####")[-1]
    elif prompt_name in ["prompt_generation_zero_shot", "prompt_generation_5_shot"]:
        lines = [line.strip() for line in text.splitlines() if line.strip()]
        text = lines[-1] if lines else text

    nums = re.findall(r"-?\d+(?:\.\d+)?", text)
    return nums[-1] if nums else ""

In [37]:
# Do not change, method to calculate accuracy from predictions and ground truth labels

def evaluate_accuracy(predictions: List[str], ground_truths: List[str]) -> float:
    correct = 0
    total = len(predictions)

    for pred, true in zip(predictions, ground_truths):
        if pred == true:
            correct += 1

    accuracy = correct / total
    return accuracy * 100

### Next, write the wrapper function where you use all the building blocks constructed above to prompt the Gemini model (5 + 10 pts)


On how to prompt Gemini, refer to the [Gemini Text Generation Handbook](https://ai.google.dev/gemini-api/docs/text-generation?lang=python).

Hint: Reading this will help you figure out how to generate multiple candidates to implement Self-Consistency.

In [38]:
def pipeline_generate(
    model_instance: Any,
    test_set: Dataset,
    prompt_function: Callable[[str], str],
    process_answer_function: Callable[[str], str],
    evaluation_function: Callable[[List[str], List[str]], float],
    self_consistency: int,
) -> float:
    """
    Args:
    model_instance (Any): The Google Gemini model instance.
    test_set (Dataset): The GSM8K test set to evaluate on.
    prompt_function (Callable): Function to generate prompts for the test set.
    process_answer_function (Callable): Function to process the model's generated answers.
    evaluation_function (Callable): Function to evaluate model's answers against the ground truth.
    self_consistency: Number of samples to run self-consistency approach on.
    If negative, 0 or 1, this implies regular prompting

    Returns:
    float: The accuracy of the model on the test set.
    """

    from collections import Counter
    import time

    random.seed(42)

    def call_model(prompt: str, temperature: float):
        generation_config = {"temperature": temperature}

        for attempt in range(5):
            try:
                try:
                    return model_instance.generate_content(
                        prompt,
                        generation_config=generation_config,
                    )
                except TypeError:
                    try:
                        return model_instance.generate_content(
                            prompt,
                            temperature=temperature,
                        )
                    except TypeError:
                        return model_instance.generate_content(prompt)

            except Exception as e:
                message = str(e)
                is_transient = (
                    "503" in message
                    or "UNAVAILABLE" in message
                    or "overloaded" in message.lower()
                )

                if not is_transient or attempt == 4:
                    raise

                wait_time = 2 ** attempt
                print(f"Retrying after Gemini error: {e}. Waiting {wait_time}s...")
                time.sleep(wait_time)

    def make_prompt(problem: str, prompt_fn: Callable[[str], str]) -> str:
        if prompt_fn.__name__ in ["prompt_generation_5_shot", "prompt_generation_5_shot_cot"]:
            return prompt_fn(problem, dataset["train"])
        return prompt_fn(problem)

    def run_single(split: Dataset, prompt_fn: Callable[[str], str], sc_runs: int) -> float:
        predictions = []
        ground_truths = []

        for sample in split:
            problem = sample["question"]
            ground_truths.append(sample["processed_answer"])
            prompt = make_prompt(problem, prompt_fn)

            if sc_runs is not None and sc_runs > 1:
                candidate_answers = []
                for _ in range(sc_runs):
                    response = call_model(prompt, temperature=0.7)
                    prediction_text = response.text if hasattr(response, "text") else str(response)
                    ans = process_answer_function(prediction_text, prompt_fn)
                    if ans != "":
                        candidate_answers.append(ans)

                vote = Counter(candidate_answers).most_common(1)[0][0] if candidate_answers else ""
                predictions.append(vote)
            else:
                response = call_model(prompt, temperature=0.0)
                prediction_text = response.text if hasattr(response, "text") else str(response)
                predictions.append(process_answer_function(prediction_text, prompt_fn))

        return evaluation_function(predictions, ground_truths)

    return run_single(test_set, prompt_function, self_consistency)


In [51]:
gsm8k_test_processed = process_gsm8k_answers(dataset['test'])

gsm8k_test_processed = Dataset.from_dict(gsm8k_test_processed[:5])

experiments = [
    ("0-shot", prompt_generation_zero_shot, 1),
    ("0-shot CoT", prompt_generation_zero_shot_cot, 1),
    ("5-shot", prompt_generation_5_shot, 1),
    ("5-shot CoT", prompt_generation_5_shot_cot, 1),
    ("My prompt", my_prompt, 1),
    ("0-shot CoT Self-Consistency", prompt_generation_zero_shot_cot, 3),
]

for name, prompt_fn, sc in experiments:
    accuracy = pipeline_generate(
        model_instance=model,
        test_set=gsm8k_test_processed,
        prompt_function=prompt_fn,
        process_answer_function=answer_processing,
        evaluation_function=evaluate_accuracy,
        self_consistency=sc,
    )
    print(f"{name}: {accuracy:.2f}%")


0-shot: 20.00%
0-shot CoT: 100.00%
5-shot: 80.00%
5-shot CoT: 80.00%
My prompt: 80.00%
0-shot CoT Self-Consistency: 100.00%


## 4.3 Complete this table based on your implementation in 3.2 and answer the following questions (5 + 5 pts)

### Round each value up to two decimal points (5 pts)

Method|Accuracy
---|---|
0-shot|20.00
0-shot CoT|100.00
5-shot|80.00
5-shot CoT|80.00
My prompt|80.00
0-shot CoT Self-Consistency|100.00


### What was the intuition behind the prompt that you designed? (2 pts)

Answer: The intuition behind my custom prompt was to reduce arithmetic and extraction errors by forcing a structured process: identify quantities/units, do concise step-by-step reasoning, verify once, and then output a strict final format. GSM8K errors often come from missing a constraint in the story or from formatting/parsing mismatches in the final answer. By explicitly asking for a one-line final answer (`Final Answer: <number>`), I also made post-processing more reliable and reduced ambiguity when extracting numeric outputs.

### What are the merits and demerits of using advanced prompting approaches like Chain of Thought or Self-Consistency? (3 pts)

Answer: Merits: Chain of Thought (CoT) can improve reasoning quality on multi-step problems by encouraging the model to break the task into intermediate steps instead of guessing the final number directly. Self-Consistency can further improve robustness by sampling multiple reasoning paths and taking a majority vote, which often cancels out single-sample mistakes. These methods are especially helpful for math word problems with several operations.

Demerits: They are more expensive and slower because they generate more tokens (CoT) and multiple responses (Self-Consistency). They can still produce confidently wrong reasoning, and majority vote does not guarantee correctness if the same misconception appears across samples. CoT can also make answer extraction harder if output formatting is inconsistent, which requires stricter post-processing.